# qBraid GPU tensor network simulator

> **Billing notice:** each GPU simulator has a **free daily allowance**: 25 circuits a day on the Free plan, 125 on Standard, and unlimited on Pro and Enterprise. A single job is one circuit, and a batch counts one per circuit. Past it, jobs are refused unless you turn on usage-based billing on your Usage page, which charges them at **5 credits per minute** of execution time. This notebook runs about ten jobs, plus one batch of four circuits.

`qbraid:qbraid:sim:gpu-tn` is qBraid's GPU **tensor network** simulator, on NVIDIA GH200 Grace
Hopper GPUs. It handles **wide, weakly entangled** circuits, up to **500 qubits**, in two ways:

- **Matrix product state (MPS)**, the default: compresses the state, so its cost depends on
  entanglement rather than width. Exact while the entanglement fits in the bond dimension, and
  approximate past it.
- **Exact contraction**: contracts the whole tensor network, exactly, for circuits small or
  shallow enough to finish.

For exact results on circuits up to 35 qubits, however entangled, use its sibling, the GPU state
vector simulator `qbraid:qbraid:sim:gpu-sv`, which has its own notebook.

This notebook covers:

1. A 100-qubit circuit on the default MPS method
2. The options that control MPS truncation, and what too much truncation looks like
3. Exact contraction
4. Reproducible sampling with `seed`
5. Running several circuits as one batch job

In [ ]:
%%capture

# "%%capture" hides the install output; comment it out if you need to debug this step.
%pip install 'qbraid[qiskit,visualization]'

## Connect to the device

`QbraidProvider` picks up your API key from `~/.qbraid/qbraidrc` or the `QBRAID_API_KEY`
environment variable. Inside qBraid Lab it is already configured for you.

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
device = provider.get_device("qbraid:qbraid:sim:gpu-tn")

device.metadata()

gpu-tn takes OpenQASM 2 or 3, and qBraid transpiles to it for you, so you can pass a Qiskit,
Cirq, Braket, pyQuil, or pytket circuit straight to `run()`.

## 100 qubits

A 100-qubit GHZ state is far past any state vector, but easy for MPS. Writing it as a chain of
neighbouring CNOTs keeps it fast:

In [ ]:
from qiskit import QuantumCircuit


def ghz(num_qubits: int) -> QuantumCircuit:
    circuit = QuantumCircuit(num_qubits)
    circuit.h(0)
    for qubit in range(num_qubits - 1):
        circuit.cx(qubit, qubit + 1)
    circuit.measure_all()
    return circuit


result = device.run(ghz(100), shots=1000).result()
counts = result.data.get_counts()

{f"{key[:8]}…{key[-8:]}": count for key, count in counts.items()}

All 100 qubits agree on every shot. A GHZ state, however wide, is only lightly entangled: any
cut through the chain carries one bit of entanglement, so MPS represents it exactly.

Each result records how it was simulated, including the truncation settings an MPS result was
sampled under:

In [ ]:
keys = (
    "numQubits",
    "simulationMethod",
    "precision",
    "maxBondDim",
    "relativeCutoff",
    "absoluteCutoff",
)
{key: result.details[key] for key in keys}

## Runtime options

gpu-tn's settings go in a **`runtime_options` dict**, not in the `run()` signature. `shots` is
the exception: pass it to `run()` directly, from 1 to 10,000.

| Option | Default | What it does |
|---|---|---|
| `method` | `"mps"` | `"mps"`: matrix product state. `"exact"`: full tensor network contraction |
| `maxBondDim` | `64` | MPS only. The largest bond dimension kept, 2 to 1024 |
| `relativeCutoff` | `1e-5` | MPS only. Singular values below this fraction of the largest are dropped |
| `absoluteCutoff` | `1e-5` | MPS only. Singular values below this value are dropped |
| `svdAlgorithm` | `"GESVDJ"` | MPS only. The SVD that truncates the bonds: `"GESVD"`, `"GESVDJ"` or `"GESVDP"` |
| `precision` | `"double"` | `"double"` (complex128) or `"single"` (complex64) |
| `seed` | none | Seeds the sampling: the same seed, circuit and options return the same counts |
| `timeout` | none | A time limit in whole seconds, at least 1. Every job also stops after 10 minutes |

## When MPS is approximate

MPS is exact while every bond stays within `maxBondDim`. A strongly entangling circuit needs
larger bonds, and past `maxBondDim` the simulator truncates, so the counts are sampled from an
approximation.

A **mirror circuit** makes that visible: random layers followed by their exact inverse. The
ideal outcome is all zeros with probability 1, so any other outcome is truncation error:

In [ ]:
import numpy as np


def mirror_circuit(num_qubits: int, layers: int, seed: int = 11) -> QuantumCircuit:
    rng = np.random.default_rng(seed)
    half = QuantumCircuit(num_qubits)
    for layer in range(layers):
        for qubit in range(num_qubits):
            half.u(*rng.uniform(0, 2 * np.pi, 3), qubit)
        for qubit in range(layer % 2, num_qubits - 1, 2):
            half.cx(qubit, qubit + 1)
    circuit = half.compose(half.inverse())
    circuit.measure_all()
    return circuit


mirror = mirror_circuit(16, 12)
zeros = "0" * 16


def share_of_zeros(options: dict) -> float:
    counts = device.run(mirror, shots=1000, runtime_options=options).result().data.get_counts()
    return counts.get(zeros, 0) / 1000


for options in ({}, {"maxBondDim": 4}, {"maxBondDim": 2}):
    print(f"{str(options):20} share of shots on all zeros: {share_of_zeros(options):.3f}")

The defaults keep this circuit exact. Squeezing the bond dimension to 4 or 2 throws away
entanglement the circuit needs, and the counts spread over many wrong outcomes.

To check whether truncation matters for your own circuit, run it again with a larger
`maxBondDim`, or with `method: "exact"` if the circuit is small enough, and compare. If the counts
agree, the smaller setting was enough. Raising `maxBondDim` costs memory and time, roughly with
the cube of the bond dimension. To truncate less without raising it, lower the cutoffs: both must
stay above 0, so send a tiny value such as `1e-12`.

## Exact contraction

`method: "exact"` contracts the full tensor network, with no truncation:

In [ ]:
print(f"exact share of shots on all zeros: {share_of_zeros({'method': 'exact'}):.3f}")

Exact contraction gets expensive quickly as circuits get wider and deeper. gpu-tn estimates the
cost first and refuses a circuit it can't finish in time, and stops any job after 10 minutes. A
stopped or refused job fails and is not charged. For wide circuits, MPS is usually the way.

## Reproducible results with `seed`

Sampling is random, so two identical runs normally give different counts. Fix `seed` and they
match exactly, with either method:

In [ ]:
seeded = [
    device.run(ghz(20), shots=1000, runtime_options={"seed": 7}).result().data.get_counts()
    for _ in range(2)
]

print(seeded[0])
print(seeded[1])
print("identical:", seeded[0] == seeded[1])

## When a job fails

An option the simulator doesn't know, or a value out of range, fails the job instead of being
ignored. The reason is on the job's status, in `status_message`:

In [ ]:
bad_job = device.run(ghz(3), shots=1000, runtime_options={"maxBondDim": 4096})
bad_job.wait_for_final_state()

status = bad_job.status()
print(status)
print(status.status_message)

## Batch jobs

gpu-tn supports **batch jobs**: pass a list of circuits with `as_batch=True` and they run as
**one job**, with one job ID and one status, and you get one set of counts per circuit, in order.
The circuits run one after another in the same GPU allocation, so a batch skips the submission and
queueing overhead that each separate job pays.

Here we run GHZ states of four different widths, from 50 to 400 qubits, in one job:

In [ ]:
widths = [50, 100, 200, 400]

batch_job = device.run([ghz(width) for width in widths], shots=1000, as_batch=True)
batch_counts = batch_job.result().data.get_counts()  # one dict per circuit

print(batch_job.id)
for width, counts in zip(widths, batch_counts):
    agree = counts.get("0" * width, 0) + counts.get("1" * width, 0)
    print(
        f"{width:>3} qubits: {len(counts)} outcomes, {agree / 1000:.0%} of shots all-zeros or all-ones"
    )

Every width keeps all its qubits in agreement, in a single job.

A few things to know about batches:

- **The daily allowance counts circuits.** This batch used four of today's free circuits, not one.
  A batch bigger than what remains of the allowance is refused whole, unless usage-based billing
  is on, in which case the whole batch runs paid.
- **One set of runtime options per batch.** `runtime_options` apply to every circuit in it, so
  compare settings with separate jobs.
- **Circuits fail on their own.** One bad circuit doesn't fail the others. A batch where some
  circuits fail is charged for its full execution time; one where none complete is free.
- **The time limit covers the whole batch.** A circuit that couldn't finish before the job's time
  limit isn't started, and comes back with a message saying to submit it again.

Without `as_batch=True`, a list of circuits is submitted as **separate jobs**, one per circuit,
and `run()` returns a list of jobs.

## Summary

- `qbraid:qbraid:sim:gpu-tn` runs wide, weakly entangled circuits, up to 500 qubits, on an NVIDIA GH200.
- MPS, the default, is exact while entanglement fits in `maxBondDim`, and approximate past it. `result.details` records the settings a result was sampled under.
- `method: "exact"` contracts the full network, for circuits small or shallow enough to finish.
- Settings go in `runtime_options`: `method`, `maxBondDim`, the cutoffs, `svdAlgorithm`, `precision`, `seed` and `timeout`.
- `as_batch=True` runs a list of circuits as one job, and each circuit counts against the allowance.
- Each GPU simulator has a free daily allowance, counted per circuit; past it, usage-based billing charges 5 credits per minute.
- For exact results on circuits up to 35 qubits, use `qbraid:qbraid:sim:gpu-sv`.

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>